# Flight Prices: Route Extraction

**Project:** When should you buy a plane ticket? Predicting U.S. domestic flight prices and price drops.
**Course:** Intro to Machine Learning, Fall 2026
**Author:** Anita Roese

**Purpose:** The `itineraries.csv` dataset used for this project is too large (~31 GB) to model directly, since it would slow down every step of the modeling process. In `01_dataset_overview`, a 1% random sample was generated for an initial exploration of the data. However, this sample cannot be used for modeling: it keeps only about 1 in 100 searches of each flight, so it is not possible to track a flight's price over time. Therefore, this notebook:

1. Selects 5–10 routes and saves **every search** for those routes into a new file, which will be used in the EDA and modeling phases.
2. Builds the `wait` label by checking, for each search, whether the same flight's price drops within the next 7 days (1 = wait, 0 = buy now).

## 1. Setup

In [1]:
import os
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
path = "itineraries.csv"                  # where the 31 GB CSV is
output_path = "data/selected_routes.csv"  # where the new route file will be saved
os.makedirs("data", exist_ok=True)       # create the folder for output_path if it doesn't exist

## 2. Choosing the routes

The routes are chosen in two steps.

### Step 1
A table of all routes is built, with one row per route (origin → destination) showing the number of rows for that route (number of searches), the number of unique flights, and the average distance.

### Step 2
From this table, three criteria are used to pick the routes:
1. **Enough data:** routes are sorted by number of searches, and routes near the top are selected. The `wait` label compares each search with the same flight's later searches, so the model needs routes whose flights were searched many times.
2. **A mix of short and long routes:** using the average distance column, some routes below the average distance and some above it are selected. If all routes had a similar distance, distance and travel duration would barely vary, and the model could not learn how they affect price.
3. **Not all from one airport:** the selected routes start at different airports. If every route started at the same airport, the results would only describe that one city instead of U.S. domestic flights in general.

The 20 most-searched routes overall:

In [3]:

duckdb.sql(f"""
    SELECT startingAirport, 
           destinationAirport,
           COUNT(*)                              AS n_searches,
           COUNT(DISTINCT legId)                 AS n_flights,
           ROUND(AVG(totalTravelDistance), 0)    AS avg_distance
    FROM '{path}'
    GROUP BY startingAirport, destinationAirport
    ORDER BY n_searches DESC
    LIMIT 20
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬────────────────────┬────────────┬───────────┬──────────────┐
│ startingAirport │ destinationAirport │ n_searches │ n_flights │ avg_distance │
│     varchar     │      varchar       │   int64    │   int64   │    double    │
├─────────────────┼────────────────────┼────────────┼───────────┼──────────────┤
│ ATL             │ LAX                │     709809 │     43756 │       2145.0 │
│ LAX             │ BOS                │     679169 │     59713 │       2711.0 │
│ LGA             │ LAX                │     677713 │     60870 │       2609.0 │
│ LAX             │ ATL                │     669609 │     42537 │       2182.0 │
│ LAX             │ LGA                │     663659 │     59968 │       2642.0 │
│ BOS             │ LAX                │     644390 │     58834 │       2704.0 │
│ LAX             │ JFK                │     625496 │     40860 │       2657.0 │
│ LAX             │ ORD                │     620576 │     42642 │       2016.0 │
│ DFW             │ LAX     

Because the top routes are mostly long LAX routes, the 20-most searched routes under 1,000 miles are listed separetely:

In [4]:
duckdb.sql(f"""
    SELECT startingAirport, 
           destinationAirport,
           COUNT(*)                              AS n_searches,
           COUNT(DISTINCT legId)                 AS n_flights,
           ROUND(AVG(totalTravelDistance), 0)    AS avg_distance
    FROM '{path}'
    GROUP BY startingAirport, destinationAirport
    HAVING avg_distance < 1000
    ORDER BY n_searches DESC
    LIMIT 20
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬────────────────────┬────────────┬───────────┬──────────────┐
│ startingAirport │ destinationAirport │ n_searches │ n_flights │ avg_distance │
│     varchar     │      varchar       │   int64    │   int64   │    double    │
├─────────────────┼────────────────────┼────────────┼───────────┼──────────────┤
│ LGA             │ ORD                │     550319 │     23215 │        765.0 │
│ ORD             │ LGA                │     546060 │     22441 │        764.0 │
│ ORD             │ JFK                │     515195 │     26749 │        947.0 │
│ LGA             │ CLT                │     512110 │     24279 │        667.0 │
│ CLT             │ LGA                │     509152 │     24930 │        666.0 │
│ IAD             │ LGA                │     497571 │     41787 │        838.0 │
│ ATL             │ BOS                │     493277 │     37196 │        996.0 │
│ BOS             │ LGA                │     483784 │     24370 │        409.0 │
│ ATL             │ LGA     

### Selected routes

Most of the busiest routes in the dataset are long, cross-country flights to or from LAX. Choosing only the top routes would therefore give a set of long LAX routes, failing criteria 2 and 3. To balance the three criteria, routes were also selected from routes with an average distance under 1,000 miles.

| Route | Avg. distance | Searches | Type | Reason |
|---|---|---|---|---|
| BOS → LGA | 409 mi | 483,784 | short | Shortest route in the set; extends the distance range downward |
| ORD → LGA | 764 mi | 546,060 | short | Second most-searched route under 1,000 miles; adds Chicago as an origin |
| CLT → ORD | 973 mi | 448,594 | short | Most-searched short route from Charlotte; ends at ORD to spread the destinations |
| DFW → LAX | 1,494 mi | 612,390 | medium | Fills the gap between short and long routes; adds Dallas as an origin |
| ATL → LAX | 2,145 mi | 709,809 | long | Most-searched route in the dataset |
| LAX → BOS | 2,711 mi | 679,169 | long | Most-searched route starting at LAX; longest route in the set |

The six routes start at six different airports and cover distances from about 400 to 2,700 miles, with about **3.5 million searches** in total.

**Limitation:** two routes end at LGA and two at LAX, so results may partly reflect pricing at those airports.

In [5]:
# Selected routes as (origin, destination), ordered from shortest to longest
routes = [
    ("BOS", "LGA"),   # 409 mi, short
    ("ORD", "LGA"),   # 764 mi, short
    ("CLT", "ORD"),   # 973 mi, short
    ("DFW", "LAX"),   # 1,494 mi, medium
    ("ATL", "LAX"),   # 2,145 mi, long
    ("LAX", "BOS"),   # 2,711 mi, long
]

## 3. Extracting the selected routes

The full dataset has about 82 million searches across all routes, but only the searches for the six selected routes are needed for the EDA and modeling. In this step, DuckDB reads the full CSV and keeps:

- **only the rows** whose origin and destination match one of the six selected routes, and
- **only the 11 columns** used in this project.

Because DuckDB streams through the file without loading it into memory, this works on a laptop even though the file is about 31 GB. The result (about 3.5 million searches) is saved to `data/selected_routes.csv`, so the following notebooks can load it in seconds instead of scanning the full file again.

> **Note:** `data/` is excluded from GitHub by `.gitignore`. To recreate `selected_routes.csv`, download `itineraries.csv` from Kaggle and run this notebook.

In [8]:
# Columns used in this project (same as notebook 01)
cols = ["legId", "searchDate", "flightDate", "startingAirport",
        "destinationAirport", "totalFare", "seatsRemaining",
        "travelDuration", "totalTravelDistance", "isNonStop",
        "isBasicEconomy"]

# Keep only the selected routes and columns, and save the result to a CSV file
duckdb.sql(f"""
    COPY (
        SELECT {", ".join(cols)}
        FROM '{path}'
        WHERE (startingAirport, destinationAirport) IN {tuple(routes)}
    ) TO '{output_path}' (HEADER)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CountBIGINT
3479806


The `WHERE (startingAirport, destinationAirport) IN (...)` condition keeps only rows whose origin–destination pair matches one of the six selected routes. The pairs are taken directly from the `routes` list, so changing a route only requires editing that list. `COPY ... TO` writes the result straight to `data/selected_routes.csv` without loading it into memory.

DuckDB reports **3,479,806 rows** written, which equals the sum of the searches for the six routes in the full dataset (483,784 + 546,060 + 448,594 + 612,390 + 709,809 + 679,169), so no rows were lost or added.

In [9]:
# Overall summary of the extracted file
duckdb.sql(f"""
    SELECT COUNT(*)                        AS rows,
           COUNT(DISTINCT legId)           AS flights,
           MIN(searchDate)                 AS first_search,
           MAX(searchDate)                 AS last_search,
           COUNT(DISTINCT startingAirport) AS airports
    FROM '{output_path}'
""").show()

┌─────────┬─────────┬──────────────┬─────────────┬──────────┐
│  rows   │ flights │ first_search │ last_search │ airports │
│  int64  │  int64  │     date     │    date     │  int64   │
├─────────┼─────────┼──────────────┼─────────────┼──────────┤
│ 3479806 │  213972 │ 2022-04-16   │ 2022-10-05  │        6 │
└─────────┴─────────┴──────────────┴─────────────┴──────────┘



*Interpretation:* The extracted file contains **3,479,806 searches** of **213,972 unique flights** from **6 origin airports**, searched between **2022-04-16** and **2022-10-05**, the same period as the full dataset. On average, each flight is observed on about **16.3 search dates** (rows ÷ flights), slightly more than the 13.7 in the full dataset. This is what makes it possible to follow a flight's price over time and build the `wait` label.

In [10]:
# Rows and flights per route, to check against the full dataset
duckdb.sql(f"""
    SELECT startingAirport,
           destinationAirport,
           COUNT(*)                           AS n_searches,
           COUNT(DISTINCT legId)              AS n_flights,
           ROUND(AVG(totalTravelDistance), 0) AS avg_distance
    FROM '{output_path}'
    GROUP BY startingAirport, destinationAirport
    ORDER BY avg_distance
""").show()

┌─────────────────┬────────────────────┬────────────┬───────────┬──────────────┐
│ startingAirport │ destinationAirport │ n_searches │ n_flights │ avg_distance │
│     varchar     │      varchar       │   int64    │   int64   │    double    │
├─────────────────┼────────────────────┼────────────┼───────────┼──────────────┤
│ BOS             │ LGA                │     483784 │     24370 │        409.0 │
│ ORD             │ LGA                │     546060 │     22441 │        764.0 │
│ CLT             │ ORD                │     448594 │     31088 │        973.0 │
│ DFW             │ LAX                │     612390 │     32604 │       1494.0 │
│ ATL             │ LAX                │     709809 │     43756 │       2145.0 │
│ LAX             │ BOS                │     679169 │     59713 │       2711.0 │
└─────────────────┴────────────────────┴────────────┴───────────┴──────────────┘



*Interpretation:* The number of searches for each route matches the values in the full dataset (Section 2), confirming that every search for the six routes was extracted. The routes range from about 409 miles (BOS → LGA) to about 2,711 miles (LAX → BOS), covering short, medium, and long distances.